# Datos del dashboard de alertas de demora
**Trabajo Final Integrador — Analítica de Negocios Avanzada (UCEMA, 2026)**

Este notebook arma los datos que muestra el dashboard (FASE 6). **No entrena ni recalcula el modelo:** solo lee las salidas de los otros dos notebooks, las une y las deja listas para mostrar.

| Insumo | Lo genera | Qué aporta |
|---|---|---|
| `generated/predicciones_test_detalle.csv` | `tp.ipynb` (6.12.4) | Datos del vuelo, clima, score, nivel del semáforo y factores SHAP |
| `generated/sintetico/sintetico_pasajeros_vuelos_test.csv` | `simulacion_impacto.ipynb` (8) | Asientos, pasajeros y pasajeros con conexión de cada vuelo |
| `generated/sintetico/sintetico_costos_esperados_por_nivel.csv` | `simulacion_impacto.ipynb` (8.1) | Tasa de retraso y costo medio de una demora por nivel, estimados con validación |
| `generated/sintetico/sintetico_comparacion_politicas_*.csv` | `simulacion_impacto.ipynb` (8) | Resultados de las políticas de acción, para la página de impacto |

**Orden de ejecución:** `tp.ipynb` → `simulacion_impacto.ipynb` → este notebook.

**Qué muestra el dashboard por vuelo:**
- **Overview:** ciudad de origen y destino, aerolínea, horario programado, clima previsto, pasajeros, ocupación y costo esperado.
- **Alertas:** score de riesgo y tasa real de retraso de su nivel, los 3 factores que explican la alerta, la acción sugerida y el costo de cada acción.

**Aclaraciones para la documentación técnica** (el dashboard no las muestra):
- Los **pasajeros, la ocupación y los costos** son **sintéticos** (supuestos documentados en `simulacion_impacto.ipynb`). En el dashboard se presentan como datos de la operación.
- El **clima** es el observado del día (Open-Meteo). En el dashboard se muestra como "clima previsto", que es lo que habría en producción (pronóstico a 5 días).
- No se usa ninguna columna de resultado del vuelo (`retraso_salida`, `retrasado`, causas): el dashboard muestra lo que operaciones sabría **antes de la salida**.

## 1. Configuración

In [1]:
import os
import json

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

# Insumos (salidas de tp.ipynb y simulacion_impacto.ipynb)
ARCHIVO_DETALLE = 'generated/predicciones_test_detalle.csv'
CARPETA_SINTETICO = 'generated/sintetico'
ARCHIVO_PASAJEROS = f'{CARPETA_SINTETICO}/sintetico_pasajeros_vuelos_test.csv'
ARCHIVO_COSTOS_NIVEL = f'{CARPETA_SINTETICO}/sintetico_costos_esperados_por_nivel.csv'
ARCHIVO_POLITICAS_VAL = f'{CARPETA_SINTETICO}/sintetico_comparacion_politicas_validacion.csv'
ARCHIVO_POLITICA_TEST = f'{CARPETA_SINTETICO}/sintetico_resultado_politica_test.csv'

# Salidas
CARPETA_DASHBOARD = 'generated/dashboard'
ARCHIVO_VUELOS = f'{CARPETA_DASHBOARD}/dashboard_vuelos.csv'
ARCHIVO_NIVELES = f'{CARPETA_DASHBOARD}/dashboard_niveles.csv'
ARCHIVO_IMPACTO = f'{CARPETA_DASHBOARD}/dashboard_impacto.csv'
ARCHIVO_JSON = f'{CARPETA_DASHBOARD}/dashboard_datos.json'

NIVELES = ['Rojo', 'Naranja', 'Amarillo', 'Verde']          # orden de prioridad en el dashboard
N_FACTORES = 3                                               # factores que se muestran por alerta
N_FACTORES_EXPORTADOS = 5                                    # factores que exporta tp.ipynb (6.12.4)

# Acción sugerida por nivel (semáforo final, tp.ipynb 6.11). La reubicación preventiva no se muestra:
# según la simulación no se paga en ningún nivel.
ACCION_SUGERIDA = {
    'Verde': 'Sin acción',
    'Amarillo': 'Avisar a los pasajeros',
    'Naranja': 'Avisar a los pasajeros y monitoreo prioritario de operaciones',
    'Rojo': 'Avisar a los pasajeros y reasignar tripulación y recursos',
}

# Umbrales para el resumen del clima: los mismos cortes del EDA y de SHAP (tp.ipynb 4.3.5 y 6.12.2)
TRAMOS_LLUVIA_MM = [(20, 'Lluvia fuerte'), (5, 'Lluvia moderada'), (1, 'Lluvia leve')]
VIENTO_FUERTE_KMH = 35
TEMPERATURA_HELADA_C = 0

## 2. Carga y unión de los insumos

El detalle de `tp.ipynb` y los pasajeros de la simulación salen de los mismos vuelos de test y **en el mismo orden** (`predicciones_test.csv`), así que se unen por posición. Antes se controla que coincidan fecha, aerolínea, origen y destino fila por fila: si no coinciden, la unión mezclaría vuelos distintos.

In [2]:
detalle = pd.read_csv(ARCHIVO_DETALLE, parse_dates=['fecha_vuelo'])
pasajeros = pd.read_csv(ARCHIVO_PASAJEROS, parse_dates=['fecha_vuelo'])
costos_nivel = pd.read_csv(ARCHIVO_COSTOS_NIVEL, index_col='nivel')

claves = ['fecha_vuelo', 'codigo_aerolinea', 'origen', 'destino']
assert len(detalle) == len(pasajeros), 'El detalle y los pasajeros tienen distinta cantidad de vuelos'
assert detalle[claves].equals(pasajeros[claves]), 'El detalle y los pasajeros no están en el mismo orden'

vuelos = pd.concat([detalle, pasajeros[['asientos', 'pasajeros', 'pasajeros_conexion']]], axis=1)
print(f"Vuelos de test: {len(vuelos):,}")
print(f"Período: {vuelos['fecha_vuelo'].min().date()} a {vuelos['fecha_vuelo'].max().date()} "
      f"({vuelos['fecha_vuelo'].nunique()} días, {len(vuelos) / vuelos['fecha_vuelo'].nunique():.0f} vuelos por día en promedio)")
print(f"Ciudades de origen: {vuelos['ciudad_origen'].nunique()}")
print("\nCostos por nivel (estimados con validación en la simulación):")
display(costos_nivel.round(3).T)

Vuelos de test: 36,998
Período: 2022-01-07 a 2022-12-31 (359 días, 103 vuelos por día en promedio)
Ciudades de origen: 88

Costos por nivel (estimados con validación en la simulación):


nivel,Verde,Amarillo,Naranja,Rojo
vuelos,47221,39055,21176,3514
tasa_retraso,0.101,0.206,0.374,0.613
sin_operativo_si_demora,7190.212,6534.405,6626.522,7470.875
sin_compensacion_pp_si_demora,2.748,1.484,1.38,1.466
sin_reubicacion_pc_si_demora,96.679,94.202,98.84,115.571
sin_ingreso_riesgo_pp_si_demora,14.138,13.565,13.835,15.215
con_operativo_si_demora,7190.212,6534.405,6626.522,5229.612
con_compensacion_pp_si_demora,2.748,1.484,1.38,0.699
con_reubicacion_pc_si_demora,96.679,94.202,98.84,80.892
con_ingreso_riesgo_pp_si_demora,14.138,12.209,12.452,10.135


## 3. Datos del vuelo para el overview

- **Ciudad:** el dataset no tiene región, así que el filtro principal del dashboard es la ciudad de origen. Se separa también el estado (`"Dallas/Fort Worth, TX"` → `TX`) por si sirve como filtro adicional.
- **Horarios:** vienen en formato hhmm (1430 = 14:30) y se pasan a texto `HH:MM`.
- **Ocupación:** pasajeros / asientos.
- **Clima previsto:** temperatura, lluvia y viento del aeropuerto de origen, más un resumen en texto con las condiciones que el EDA y SHAP mostraron como de riesgo: lluvia (desde 1 mm, por tramos), viento de más de 35 km/h y temperatura bajo cero.

In [3]:
def hora_texto(hhmm):
    hhmm = hhmm.astype(int)
    return (hhmm // 100 % 24).astype(str).str.zfill(2) + ':' + (hhmm % 100).astype(str).str.zfill(2)


def franja_horaria(hora):
    return pd.cut(hora, bins=[0, 6, 12, 18, 24], labels=['Madrugada', 'Mañana', 'Tarde', 'Noche'],
                  right=False, include_lowest=True).astype(str)


def resumen_clima(fila):
    alertas = [texto for mm, texto in TRAMOS_LLUVIA_MM if fila['precipitacion'] >= mm][:1]
    if fila['viento_max'] > VIENTO_FUERTE_KMH:
        alertas.append('Viento fuerte')
    if fila['temperatura_media'] <= TEMPERATURA_HELADA_C:
        alertas.append('Temperatura bajo cero')
    return ' · '.join(alertas) if alertas else 'Sin alertas climáticas'


vuelos['vuelo'] = vuelos['codigo_aerolinea'] + ' ' + vuelos['numero_vuelo'].astype(str)
vuelos['estado_origen'] = vuelos['ciudad_origen'].str.split(', ').str[-1]
vuelos['estado_destino'] = vuelos['ciudad_destino'].str.split(', ').str[-1]
vuelos['salida_programada'] = hora_texto(vuelos['hora_salida_programada'])
vuelos['llegada_programada'] = hora_texto(vuelos['hora_llegada_programada'])
vuelos['franja_horaria'] = franja_horaria(vuelos['hora_prog'])
vuelos['ocupacion'] = vuelos['pasajeros'] / vuelos['asientos']
vuelos['clima_resumen'] = vuelos.apply(resumen_clima, axis=1)

print("Resumen del clima previsto (% de vuelos):")
display((vuelos['clima_resumen'].value_counts(normalize=True) * 100).round(1).to_frame('%'))
display(vuelos[['vuelo', 'ciudad_origen', 'ciudad_destino', 'salida_programada', 'llegada_programada',
                'pasajeros', 'ocupacion', 'clima_resumen']].head())

Resumen del clima previsto (% de vuelos):


,%
clima_resumen,
Sin alertas climáticas,62.8
Lluvia leve,12.6
Lluvia moderada,11.8
Temperatura bajo cero,5.3
Lluvia fuerte,3.1
Viento fuerte,1.6
Lluvia leve · Temperatura bajo cero,0.8
Lluvia moderada · Viento fuerte,0.6
Lluvia leve · Viento fuerte,0.5


,vuelo,ciudad_origen,ciudad_destino,salida_programada,llegada_programada,pasajeros,ocupacion,clima_resumen
0,F9 518,"Denver, CO","Philadelphia, PA",00:36,06:04,127,0.793750,Sin alertas climáticas
1,AA 418,"Boston, MA","Charlotte, NC",05:40,08:17,131,0.818750,Lluvia moderada · Temperatura bajo cero
2,AA 944,"St. Louis, MO","Miami, FL",05:40,09:26,125,0.781250,Temperatura bajo cero
3,DL 2235,"San Jose, CA","Salt Lake City, UT",05:49,08:54,128,0.800000,Sin alertas climáticas
4,OO 3676,"Ontario, CA","Seattle, WA",06:00,09:08,66,0.868421,Sin alertas climáticas


## 4. Riesgo: score y tasa real del nivel

El modelo devuelve una probabilidad que **ordena bien el riesgo pero no está calibrada** (`tp.ipynb`, 6.12): un vuelo con 0,82 no tiene un 82% de chances de demorarse. Por eso el dashboard no la muestra como porcentaje:

- **Score de riesgo (0 a 100):** la probabilidad del modelo × 100, redondeada hacia abajo. Así el score coincide con los cortes del semáforo (40, 60 y 80): un vuelo con 0,799 es naranja y tiene score 79, no 80. Sirve para ordenar los vuelos dentro de cada nivel.
- **Tasa real del nivel:** qué porcentaje de los vuelos de ese nivel se demoró en **validación**. Es el número que sí se puede leer como frecuencia ("en este nivel se demora el 61% de los vuelos").

El orden de prioridad es: primero el nivel (rojo → verde) y, dentro de cada nivel, el score de mayor a menor.

In [4]:
vuelos['score_riesgo'] = np.floor(vuelos['proba_retraso'] * 100).astype(int)
vuelos['nivel'] = vuelos['nivel_semaforo']
vuelos['prioridad_nivel'] = vuelos['nivel'].map({n: i + 1 for i, n in enumerate(NIVELES)})
vuelos['tasa_real_nivel'] = vuelos['nivel'].map(costos_nivel['tasa_retraso'])

resumen_riesgo = vuelos.groupby('nivel').agg(
    vuelos=('nivel', 'size'), score_min=('score_riesgo', 'min'), score_max=('score_riesgo', 'max'),
    tasa_real_nivel=('tasa_real_nivel', 'first')).reindex(NIVELES)
resumen_riesgo['% de vuelos'] = (resumen_riesgo['vuelos'] / len(vuelos) * 100).round(1)
display(resumen_riesgo)

,vuelos,score_min,score_max,tasa_real_nivel,% de vuelos
nivel,,,,,
Rojo,1096,80,97,0.613261,3.0
Naranja,7091,60,79,0.374008,19.2
Amarillo,12868,40,59,0.206299,34.8
Verde,15943,7,39,0.100887,43.1


## 5. Factores que explican cada alerta (SHAP)

`tp.ipynb` exporta los 5 aportes SHAP positivos más altos de cada vuelo. Acá se traducen a lenguaje de operaciones, con el valor de la variable cuando ayuda a entender el factor (por ejemplo, *"Hora programada de salida (18 h)"* o *"Historial de retrasos de la aerolínea en este aeropuerto (37,8%)"*).

Las variables **indicadoras** (aerolínea, franja horaria, temporada alta, low cost) solo se muestran cuando valen 1. Si valen 0, su aporte significa algo como "no es Delta", que no le sirve a operaciones para decidir. Después de descartarlas se toman los 3 primeros factores.

Las descripciones son las mismas de `DESCRIPCION_FEATURES` en `tp.ipynb` (6.12.3).

In [5]:
DESCRIPCION_FEATURES = {
    'hora_prog': 'Hora programada de salida',
    'lowcost_x_hora': 'Aerolínea low cost en horario tardío',
    'tasa_hist_aerolinea_origen': 'Historial de retrasos de la aerolínea en este aeropuerto',
    'tasa_reciente_aerolinea': 'Retrasos recientes de la aerolínea (7 días)',
    'tasa_reciente_origen': 'Retrasos recientes del aeropuerto de origen (7 días)',
    'tasa_hist_origen': 'Historial de retrasos del aeropuerto de origen',
    'tasa_hist_destino': 'Historial de retrasos del aeropuerto de destino',
    'tasa_hist_ruta': 'Historial de retrasos de la ruta',
    'dia_semana': 'Día de la semana',
    'mes': 'Mes',
    'temporada_alta': 'Temporada alta (junio, julio, diciembre)',
    'temperatura_media': 'Temperatura media prevista en el origen',
    'precipitacion': 'Lluvia prevista en el origen',
    'tramo_lluvia': 'Lluvia prevista en el origen',
    'lluvia_x_hora': 'Lluvia prevista combinada con horario tardío',
    'viento_max': 'Viento máximo previsto en el origen',
    'distancia': 'Distancia del vuelo',
    'duracion_programada': 'Duración programada',
    'vuelos_programados_dia_origen': 'Vuelos programados en el origen ese día',
    'vuelos_programados_hora_origen': 'Vuelos programados en el origen a esa hora',
    'es_low_cost': 'Aerolínea low cost',
}
DIAS = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']
MESES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre',
         'octubre', 'noviembre', 'diciembre']
TRAMOS_LLUVIA = ['sin lluvia', 'leve', 'moderada', 'fuerte']
NOMBRE_AEROLINEA = vuelos.drop_duplicates('codigo_aerolinea').set_index('codigo_aerolinea')['aerolinea']
INDICADORAS_SIMPLES = {'temporada_alta', 'es_low_cost'}


def numero(x, decimales=1):
    return f'{x:,.{decimales}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')


def texto_factor(variable, valor):
    # Descripción del factor para operaciones, o None si es una indicadora que vale 0
    if pd.isna(variable):
        return None
    es_indicadora = variable.startswith(('aero_', 'franja_')) or variable in INDICADORAS_SIMPLES
    if es_indicadora:
        if valor != 1:
            return None
        if variable.startswith('aero_'):
            return f"Aerolínea: {NOMBRE_AEROLINEA.get(variable[5:], variable[5:])}"
        if variable.startswith('franja_'):
            return f"Franja horaria: {variable[7:]}"
        return DESCRIPCION_FEATURES[variable]
    descripcion = DESCRIPCION_FEATURES.get(variable, variable)
    if variable in ('hora_prog', 'lowcost_x_hora'):
        return f"{descripcion} ({int(valor)} h)"
    if variable.startswith('tasa_'):
        return f"{descripcion} ({numero(valor * 100)}%)"
    if variable == 'dia_semana':
        return f"{descripcion}: {DIAS[int(valor)]}"
    if variable == 'mes':
        return f"{descripcion}: {MESES[int(valor) - 1]}"
    if variable == 'tramo_lluvia':
        return f"{descripcion}: {TRAMOS_LLUVIA[int(valor)]}"
    unidades = {'temperatura_media': ' °C', 'precipitacion': ' mm', 'viento_max': ' km/h', 'distancia': ' millas',
                'duracion_programada': ' min', 'vuelos_programados_dia_origen': ' vuelos',
                'vuelos_programados_hora_origen': ' vuelos'}
    if variable in unidades:
        return f"{descripcion} ({numero(valor, 0 if variable not in ('temperatura_media', 'precipitacion') else 1)}{unidades[variable]})"
    return descripcion


def factores_legibles(fila):
    textos = [texto_factor(fila[f'factor_{k}_variable'], fila[f'factor_{k}_valor'])
              for k in range(1, N_FACTORES_EXPORTADOS + 1)]
    textos = [t for t in textos if t is not None][:N_FACTORES]
    return textos + [None] * (N_FACTORES - len(textos))


factores = pd.DataFrame(vuelos.apply(factores_legibles, axis=1).tolist(),
                        columns=[f'factor_{k}' for k in range(1, N_FACTORES + 1)], index=vuelos.index)
vuelos[factores.columns] = factores

alertados = vuelos['nivel'] != 'Verde'
print(f"Vuelos alertados sin ningún factor: {factores.loc[alertados, 'factor_1'].isna().sum()}")
print(f"Vuelos alertados con menos de {N_FACTORES} factores: {factores.loc[alertados].isna().any(axis=1).sum()}")
print("\nFactor principal de los vuelos en rojo (% de los vuelos en rojo):")
factor_principal = vuelos.loc[vuelos['nivel'] == 'Rojo', 'factor_1'].str.replace(r' \(.*\)$', '', regex=True)
display((factor_principal.value_counts(normalize=True) * 100).round(1).head(8).to_frame('%'))
display(vuelos.loc[vuelos['nivel'] == 'Rojo'].nlargest(3, 'score_riesgo')[['vuelo', 'score_riesgo', 'factor_1', 'factor_2', 'factor_3']])

Vuelos alertados sin ningún factor: 0
Vuelos alertados con menos de 3 factores: 3

Factor principal de los vuelos en rojo (% de los vuelos en rojo):


,%
factor_1,
Hora programada de salida,39.2
Aerolínea low cost en horario tardío,36.3
Temperatura media prevista en el origen,11.9
Historial de retrasos de la aerolínea en este aeropuerto,4.1
Lluvia prevista combinada con horario tardío,3.4
Viento máximo previsto en el origen,3.0
Retrasos recientes de la aerolínea,1.1
Aerolínea: Frontier Airlines Inc.,0.8


,vuelo,score_riesgo,factor_1,factor_2,factor_3
36234,WN 3459,97,"Temperatura media prevista en el origen (-23,9...",Aerolínea low cost en horario tardío (21 h),Historial de retrasos de la aerolínea en este ...
36361,WN 3516,96,"Temperatura media prevista en el origen (-14,7...",Historial de retrasos de la aerolínea en este ...,"Temporada alta (junio, julio, diciembre)"
36375,WN 885,96,"Temperatura media prevista en el origen (-14,7...",Aerolínea low cost en horario tardío (15 h),Viento máximo previsto en el origen (41 km/h)


## 6. Acciones sugeridas y costo esperado

**Costo esperado del vuelo.** Antes de la salida no se sabe si el vuelo se va a demorar, así que el costo se calcula como **tasa real del nivel × costo medio de una demora en ese nivel**, con los promedios que la simulación estimó en validación (`simulacion_impacto.ipynb`, 8.1). Los rubros que dependen de los pasajeros se multiplican por los pasajeros de cada vuelo:

| Rubro | Cálculo por vuelo |
|---|---|
| Costo operativo | tasa del nivel × costo operativo medio de una demora del nivel |
| Compensación | tasa del nivel × pasajeros × compensación media por pasajero |
| Reubicación por conexión perdida | tasa del nivel × pasajeros con conexión × costo medio por pasajero con conexión |

**Acciones y su costo.** Son las del semáforo final, con los costos de la simulación:
- **Aviso a los pasajeros** (desde amarillo): USD 0,10 por pasajero.
- **Reasignación de tripulación y recursos** (rojo): USD 1.500 por vuelo. Reduce la demora esperada según la efectividad de la simulación, y con eso baja el costo esperado del vuelo.

**Resultado por vuelo:**
- **Costo esperado sin acción** y **con acción** (este último incluye el costo de las acciones).
- **Ahorro neto esperado:** costo sin acción − costo con acción. En amarillo y naranja es levemente negativo (el costo del aviso): el beneficio del aviso no es un ahorro de costos, sino el ingreso futuro protegido.
- **Ingreso futuro protegido:** cuánto ingreso futuro se deja de poner en riesgo por la pérdida de fidelidad del pasajero. Como en la simulación, se muestra aparte y no se suma al ahorro.

In [6]:
t = costos_nivel.loc[vuelos['nivel']].reset_index(drop=True)
t.index = vuelos.index
pax, pax_con, tasa = vuelos['pasajeros'], vuelos['pasajeros_conexion'], t['tasa_retraso']


def costo_esperado(escenario):
    # Costo esperado de la demora por rubro (USD), sin o con las acciones del nivel
    return pd.DataFrame({
        'operativo': tasa * t[f'{escenario}_operativo_si_demora'],
        'compensacion': tasa * pax * t[f'{escenario}_compensacion_pp_si_demora'],
        'reubicacion': tasa * pax_con * t[f'{escenario}_reubicacion_pc_si_demora'],
    })


sin_accion, con_accion = costo_esperado('sin'), costo_esperado('con')
vuelos[['costo_esperado_operativo', 'costo_esperado_compensacion', 'costo_esperado_reubicacion']] = sin_accion.to_numpy()
vuelos['costo_esperado_sin_accion'] = sin_accion.sum(axis=1)

vuelos['accion_sugerida'] = vuelos['nivel'].map(ACCION_SUGERIDA)
vuelos['aviso_pasajeros'] = t['aviso'].astype(bool)
vuelos['costo_aviso_pasajeros'] = np.where(vuelos['aviso_pasajeros'], pax * t['costo_aviso_por_pasajero'], 0.0)
vuelos['reasignacion_recursos'] = t['reasignacion'].astype(bool)
vuelos['costo_reasignacion_recursos'] = np.where(vuelos['reasignacion_recursos'], t['costo_reasignacion_por_vuelo'], 0.0)
vuelos['costo_acciones'] = vuelos['costo_aviso_pasajeros'] + vuelos['costo_reasignacion_recursos']
vuelos['costo_esperado_con_accion'] = con_accion.sum(axis=1) + vuelos['costo_acciones']
vuelos['ahorro_neto_esperado'] = vuelos['costo_esperado_sin_accion'] - vuelos['costo_esperado_con_accion']

vuelos['ingreso_en_riesgo'] = tasa * pax * t['sin_ingreso_riesgo_pp_si_demora']
vuelos['ingreso_protegido'] = vuelos['ingreso_en_riesgo'] - tasa * pax * t['con_ingreso_riesgo_pp_si_demora']

columnas_costo = ['costo_esperado_sin_accion', 'costo_acciones', 'costo_esperado_con_accion',
                  'ahorro_neto_esperado', 'ingreso_protegido']
print("Promedio por vuelo según el nivel (USD):")
display(vuelos.groupby('nivel')[columnas_costo].mean().reindex(NIVELES).round(1))

Promedio por vuelo según el nivel (USD):


,costo_esperado_sin_accion,costo_acciones,costo_esperado_con_accion,ahorro_neto_esperado,ingreso_protegido
nivel,,,,,
Rojo,6583.4,1512.9,6095.2,488.2,403.3
Naranja,3482.3,12.4,3494.7,-12.4,64.2
Amarillo,1836.7,11.4,1848.1,-11.4,31.8
Verde,969.4,0.0,969.4,0.0,0.0


## 7. Controles

1. **Coherencia con la simulación:** el ahorro neto esperado por vuelo, promediado en todos los vuelos de test, tiene que ser parecido al que midió la simulación para la política elegida en test (`simulacion_impacto.ipynb`, 8). No es idéntico, porque acá se usan promedios por nivel estimados con validación y la simulación usa la demora real de cada vuelo.
2. **Sin columnas de resultado:** ninguna columna del dashboard puede depender de lo que pasó con el vuelo.
3. **Sin nulos** en las columnas que el dashboard necesita (los factores pueden faltar en los vuelos verdes).

In [7]:
politica_test = pd.read_csv(ARCHIVO_POLITICA_TEST, index_col='politica')
politica_elegida = politica_test.index[0]
ahorro_simulacion = politica_test.loc[politica_elegida, 'Ahorro neto operativo (USD/vuelo)']
print(f"Política: {politica_elegida}")
print(f"Ahorro neto por vuelo en test según la simulación (demora real):      USD {ahorro_simulacion:.2f}")
print(f"Ahorro neto esperado por vuelo en test según el dashboard (promedios):  USD {vuelos['ahorro_neto_esperado'].mean():.2f}")

columnas_dashboard = [
    # Identificación y overview
    'fecha_vuelo', 'vuelo', 'numero_vuelo', 'aerolinea', 'codigo_aerolinea', 'modelo_negocio',
    'origen', 'ciudad_origen', 'estado_origen', 'destino', 'ciudad_destino', 'estado_destino',
    'salida_programada', 'llegada_programada', 'hora_prog', 'franja_horaria', 'distancia',
    'temperatura_media', 'precipitacion', 'viento_max', 'clima_resumen',
    'asientos', 'pasajeros', 'pasajeros_conexion', 'ocupacion',
    # Riesgo y factores
    'score_riesgo', 'nivel', 'prioridad_nivel', 'tasa_real_nivel', 'factor_1', 'factor_2', 'factor_3',
    # Acciones y costos
    'accion_sugerida', 'aviso_pasajeros', 'costo_aviso_pasajeros',
    'reasignacion_recursos', 'costo_reasignacion_recursos', 'costo_acciones',
    'costo_esperado_operativo', 'costo_esperado_compensacion', 'costo_esperado_reubicacion',
    'costo_esperado_sin_accion', 'costo_esperado_con_accion', 'ahorro_neto_esperado',
    'ingreso_en_riesgo', 'ingreso_protegido',
]
columnas_resultado = ['retraso_salida', 'retrasado', 'hora_salida_real', 'retraso_llegada', 'cancelado', 'desviado']
assert not [c for c in vuelos.columns if c in columnas_resultado or c.startswith('retraso_por_')], \
    'Hay columnas de resultado del vuelo'

dashboard = (vuelos[columnas_dashboard]
             .sort_values(['fecha_vuelo', 'prioridad_nivel', 'score_riesgo'], ascending=[True, True, False])
             .reset_index(drop=True))
nulos = dashboard.drop(columns=['factor_1', 'factor_2', 'factor_3']).isna().sum()
assert nulos.sum() == 0, f"Columnas con nulos: {nulos[nulos > 0].to_dict()}"
print(f"\nTabla del dashboard: {dashboard.shape[0]:,} vuelos x {dashboard.shape[1]} columnas, sin nulos")

Política: 4 niveles (reasignación >= 0.80) (test)
Ahorro neto por vuelo en test según la simulación (demora real):      USD 9.54
Ahorro neto esperado por vuelo en test según el dashboard (promedios):  USD 8.13

Tabla del dashboard: 36,998 vuelos x 46 columnas, sin nulos


**Lectura de los controles.**
- **El ahorro esperado tiene el mismo signo y el mismo orden que el de la simulación:** USD 8,13 por vuelo en el dashboard, contra USD 9,54 en la simulación (política de 4 niveles en test).
- **La diferencia se explica porque el dashboard usa lo que se sabría antes de la salida:** la tasa de retraso y la demora media de cada nivel estimadas en **validación**. Por ejemplo, el rojo tiene una tasa del 61,3% en validación. La simulación, en cambio, usa la demora real de cada vuelo de test, y en test el rojo se demoró más (64,1%, `tp.ipynb` 6.11). Con más demoras en rojo, la reasignación evita más costo y el ahorro real sale algo mayor.
- **Por nivel, el único con ahorro neto positivo es el rojo** (unos USD 490 por vuelo): es el único donde la reasignación evita más costo esperado del que cuesta. En amarillo y naranja el ahorro neto es el costo del aviso (unos −USD 12 por vuelo), y su beneficio aparece como ingreso futuro protegido. Es la misma conclusión de la simulación (sección 8).
- La tabla final no tiene columnas de resultado del vuelo ni nulos, salvo en los factores: 3 vuelos alertados tienen menos de 3 factores, porque el resto de sus aportes positivos son indicadoras que valen 0.

## 8. Tablas de resumen

- **Niveles:** cuántos vuelos hay en cada nivel, su tasa real, la acción sugerida y los costos promedio. Es la leyenda del semáforo.
- **Impacto:** la comparación de políticas de la simulación (validación) y el resultado de la política elegida en test, para la página de impacto.

In [8]:
niveles = dashboard.groupby('nivel').agg(
    vuelos=('nivel', 'size'),
    tasa_real_nivel=('tasa_real_nivel', 'first'),
    accion_sugerida=('accion_sugerida', 'first'),
    score_min=('score_riesgo', 'min'),
    score_max=('score_riesgo', 'max'),
    pasajeros=('pasajeros', 'sum'),
    costo_esperado_sin_accion_medio=('costo_esperado_sin_accion', 'mean'),
    costo_acciones_medio=('costo_acciones', 'mean'),
    ahorro_neto_esperado_medio=('ahorro_neto_esperado', 'mean'),
    ingreso_protegido_medio=('ingreso_protegido', 'mean'),
).reindex(NIVELES).reset_index()
niveles['pct_vuelos'] = niveles['vuelos'] / niveles['vuelos'].sum()
display(niveles.round(3))

impacto = pd.concat([
    pd.read_csv(ARCHIVO_POLITICAS_VAL, index_col='politica').assign(conjunto='Validación'),
    politica_test.assign(conjunto='Test'),
]).reset_index()
impacto['politica_elegida'] = impacto['politica'].str.startswith(politica_elegida.replace(' (test)', ''))
display(impacto.round(2))

,nivel,vuelos,tasa_real_nivel,accion_sugerida,score_min,score_max,pasajeros,costo_esperado_sin_accion_medio,costo_acciones_medio,ahorro_neto_esperado_medio,ingreso_protegido_medio,pct_vuelos
0,Rojo,1096,0.613,Avisar a los pasajeros y reasignar tripulación...,80,97,141892,6583.351,1512.946,488.163,403.307,0.030
1,Naranja,7091,0.374,Avisar a los pasajeros y monitoreo prioritario...,60,79,880060,3482.312,12.411,-12.411,64.220,0.192
2,Amarillo,12868,0.206,Avisar a los pasajeros,40,59,1461737,1836.745,11.359,-11.359,31.789,0.348
3,Verde,15943,0.101,Sin acción,7,39,1709112,969.375,0.000,0.000,0.000,0.431


,politica,% vuelos con reasignación,Precision reasignación (%),Ahorro en demoras (USD/vuelo),Costo de acciones (USD/vuelo),Ahorro neto operativo (USD/vuelo),Ingreso protegido (USD/vuelo),Horas de espera evitadas (por vuelo),conjunto,politica_elegida
0,Actual (3 niveles),22.25,40.81,317.29,766.85,-449.57,67.09,9.12,Validación,False
1,Solo aviso,0.00,NaN,0.00,6.78,-6.78,27.23,6.46,Validación,False
2,4 niveles (reasignación >= 0.70),10.94,48.69,164.60,170.83,-6.23,51.00,8.12,Validación,False
3,4 niveles (reasignación >= 0.75),6.51,54.58,110.81,104.38,6.43,42.99,7.58,Validación,False
4,4 niveles (reasignación >= 0.80),3.17,61.33,63.29,54.28,9.01,36.18,7.10,Validación,True
5,4 niveles (reasignación >= 0.85),1.12,70.22,28.01,23.62,4.39,31.22,6.74,Validación,False
6,4 niveles (reasignación >= 0.90),0.17,80.00,5.50,9.35,-3.85,28.00,6.52,Validación,False
7,4 niveles (reasignación >= 0.80) (test),2.96,64.05,60.69,51.15,9.54,35.08,6.91,Test,True
8,"Actual (3 niveles) (test, referencia)",22.13,41.07,309.17,763.21,-454.04,64.61,8.91,Test,False


## 9. Exportación

Se guardan dos formatos, para no atar los datos a una herramienta:
- **CSV** (una tabla por archivo): para cargarlos en una hoja de cálculo o en una herramienta de BI.
- **JSON** (todo junto): para un dashboard en HTML. Los vuelos van en formato de columnas (`columnas` + `filas`) para que el archivo pese menos, y los números se redondean a lo que se muestra.

In [9]:
os.makedirs(CARPETA_DASHBOARD, exist_ok=True)
dashboard.to_csv(ARCHIVO_VUELOS, index=False)
niveles.to_csv(ARCHIVO_NIVELES, index=False)
impacto.to_csv(ARCHIVO_IMPACTO, index=False)

redondeo = {'ocupacion': 3, 'tasa_real_nivel': 3, 'temperatura_media': 1, 'precipitacion': 1, 'viento_max': 1}
vuelos_json = dashboard.copy()
vuelos_json['fecha_vuelo'] = vuelos_json['fecha_vuelo'].dt.strftime('%Y-%m-%d')
for col in vuelos_json.select_dtypes('float').columns:
    vuelos_json[col] = vuelos_json[col].round(redondeo.get(col, 2))
vuelos_json = vuelos_json.astype(object).where(vuelos_json.notna(), None)

datos = {
    'metadatos': {
        'descripcion': 'Alertas de demora de vuelos: semáforo de riesgo, factores y acciones sugeridas',
        'periodo': [vuelos_json['fecha_vuelo'].min(), vuelos_json['fecha_vuelo'].max()],
        'politica': politica_elegida,
        'niveles': NIVELES,
        'acciones': ACCION_SUGERIDA,
    },
    'vuelos': {'columnas': list(vuelos_json.columns), 'filas': vuelos_json.to_numpy().tolist()},
    'niveles': json.loads(niveles.round(4).to_json(orient='records', force_ascii=False)),
    'impacto': json.loads(impacto.round(4).to_json(orient='records', force_ascii=False)),
}
with open(ARCHIVO_JSON, 'w', encoding='utf-8') as f:
    json.dump(datos, f, ensure_ascii=False, separators=(',', ':'))

for archivo in [ARCHIVO_VUELOS, ARCHIVO_NIVELES, ARCHIVO_IMPACTO, ARCHIVO_JSON]:
    print(f"{archivo}: {os.path.getsize(archivo) / 1e6:.1f} MB")

generated/dashboard/dashboard_vuelos.csv: 19.6 MB
generated/dashboard/dashboard_niveles.csv: 0.0 MB
generated/dashboard/dashboard_impacto.csv: 0.0 MB
generated/dashboard/dashboard_datos.json: 17.1 MB
